# Lab 04 — NLP with Generative AI

## Business Context

In the previous lab we built a classic NLP pipeline for **ShopSmart**. It worked, but it had clear limits:
- It couldn't handle nuance, sarcasm, or context
- It required a labelled dataset to train
- Each new task needed a new model

Today we use **Large Language Models (LLMs)** to:
1. Classify sentiment **without any training data** (zero-shot)
2. Summarise batches of customer reviews automatically
3. Extract structured information from free text
4. Build a simple Q&A assistant over the reviews

---

## Learning Objectives

- Understand what a Large Language Model is and how to call one via API
- Write effective prompts: zero-shot, few-shot, chain-of-thought
- Extract structured data (JSON) from LLM responses
- Compare LLM-based NLP with classic NLP
- Understand key concepts: tokens, temperature, system prompts

---

## Setup

In [ ]:
# Install the Anthropic SDK if needed
# !pip install anthropic python-dotenv

In [ ]:
import os
import json
import pandas as pd
import anthropic

# Load your API key from an environment variable or .env file
# Option 1: export ANTHROPIC_API_KEY=sk-ant-...
# Option 2: use python-dotenv
# from dotenv import load_dotenv; load_dotenv()

client = anthropic.Anthropic(api_key=os.environ.get('ANTHROPIC_API_KEY'))
MODEL = 'claude-haiku-4-5-20251001'  # Fast and cost-effective for this workshop

print('Client ready. Model:', MODEL)

### A helper function to call the model

In [ ]:
def ask(prompt: str, system: str = None, temperature: float = 0.2) -> str:
    """Send a prompt to Claude and return the text response."""
    kwargs = {
        'model': MODEL,
        'max_tokens': 1024,
        'temperature': temperature,
        'messages': [{'role': 'user', 'content': prompt}]
    }
    if system:
        kwargs['system'] = system

    response = client.messages.create(**kwargs)
    return response.content[0].text

---

## Part 1 — Understanding Prompts

### 1.1 A minimal prompt

In [ ]:
response = ask("What is the capital of France?")
print(response)

### 1.2 The role of `temperature`

- `temperature = 0` → Deterministic, always the same answer. Good for structured tasks.
- `temperature = 1` → More creative and varied. Good for generation tasks.

In [ ]:
prompt = "Write a one-sentence tagline for an online shop."

print('=== temperature=0.0 ===')
for _ in range(3):
    print(ask(prompt, temperature=0.0))

print()
print('=== temperature=1.0 ===')
for _ in range(3):
    print(ask(prompt, temperature=1.0))

### 1.3 System prompts

A **system prompt** defines the model's role and behaviour for the entire conversation.

In [ ]:
system = """
You are a customer support assistant for ShopSmart, an e-commerce company.
You are helpful, concise, and friendly. You only answer questions about orders,
products, and deliveries. For anything else, politely decline.
"""

print(ask("What's the best way to track my order?", system=system))
print()
print(ask("Can you help me write my CV?", system=system))

---

## Part 2 — Zero-Shot Sentiment Analysis

In classic NLP we needed a **labelled dataset** to train a model. With LLMs we can classify text without any training examples.

In [ ]:
reviews = [
    "Absolutely love this product! Delivery was super fast and packaging was perfect.",
    "Terrible experience. Product broke after two days. Very disappointed.",
    "It's okay I guess. Arrived on time but quality could be better.",
    "Not bad for the price, but the instructions are confusing.",
    "Shocking — they sent me the wrong item and still haven't refunded me after 3 weeks!"
]

def classify_sentiment(review: str) -> dict:
    prompt = f"""
Classify the sentiment of the following customer review.
Respond with ONLY a JSON object with these keys:
  - "sentiment": one of "positive", "negative", or "neutral"
  - "confidence": a number from 0 to 1
  - "reason": one sentence explaining your choice

Review: "{review}"
"""
    raw = ask(prompt, temperature=0.0)
    # Extract JSON from response
    json_str = raw[raw.find('{'):raw.rfind('}')+1]
    return json.loads(json_str)

results = []
for review in reviews:
    result = classify_sentiment(review)
    result['review'] = review
    results.append(result)

df_results = pd.DataFrame(results)[['review', 'sentiment', 'confidence', 'reason']]
df_results

---

## Part 3 — Few-Shot Prompting

Providing **examples** in the prompt improves accuracy on specific or nuanced tasks.

In [ ]:
def classify_category(review: str) -> str:
    prompt = f"""
Classify the customer review into one of these categories:
  - DELIVERY: issues or praise about shipping/timing
  - PRODUCT_QUALITY: comments about product quality, durability, appearance
  - CUSTOMER_SUPPORT: interactions with support team
  - PRICE_VALUE: comments about pricing and value for money
  - OTHER

Here are some examples:

Review: "Arrived in 2 days, super fast!"
Category: DELIVERY

Review: "The plastic feels really cheap and broke quickly."
Category: PRODUCT_QUALITY

Review: "Support was rude and refused a refund."
Category: CUSTOMER_SUPPORT

Review: "A bit expensive but worth every cent."
Category: PRICE_VALUE

Now classify this review. Reply with ONLY the category name.
Review: "{review}"
Category:"""
    return ask(prompt, temperature=0.0).strip()

test_reviews = [
    "The box arrived crushed and the item inside was broken.",
    "Way too expensive for what you get.",
    "The agent I spoke to was incredibly patient and solved everything.",
    "Sturdy, well-built, exactly what I expected.",
]

for r in test_reviews:
    cat = classify_category(r)
    print(f'[{cat}] {r}')

---

## Part 4 — Summarisation

The marketing team wants a weekly digest of customer feedback.

In [ ]:
batch_reviews = """
1. Absolutely love this product! Delivery was super fast and packaging was perfect.
2. Great quality, exactly as described. Will definitely buy again from ShopSmart.
3. Terrible experience. Product broke after two days. Very disappointed.
4. Delivery took 3 weeks and the item arrived damaged. Unacceptable!
5. Customer service team was incredibly helpful and resolved my issue quickly.
6. Cheap quality, not worth the price at all. Waste of money.
7. Amazing value for money. The product exceeded my expectations!
8. The product stopped working after one week. No response from support.
"""

summary_prompt = f"""
You are a customer insights analyst at ShopSmart.

Below are 8 customer reviews from this week. Write a concise report (max 150 words) that includes:
1. Overall sentiment summary (positive / mixed / negative)
2. Top 2 things customers praise
3. Top 2 complaints
4. One actionable recommendation for the operations team

Reviews:
{batch_reviews}
"""

summary = ask(summary_prompt, temperature=0.3)
print(summary)

---

## Part 5 — Structured Extraction

Extract structured data from free-form text — a very common enterprise use case.

In [ ]:
def extract_review_data(review: str) -> dict:
    prompt = f"""
Extract structured information from this customer review.
Return ONLY a valid JSON object with these keys:
  - "sentiment": "positive" | "negative" | "neutral"
  - "topics": list of topics mentioned (e.g. ["delivery", "product quality"])
  - "urgency": "high" | "medium" | "low" (does this need urgent follow-up?)
  - "suggested_action": one brief action for the support team (or null if not needed)

Review: "{review}"
"""
    raw = ask(prompt, temperature=0.0)
    json_str = raw[raw.find('{'):raw.rfind('}')+1]
    return json.loads(json_str)

test_review = (
    "I ordered a laptop stand 3 weeks ago and it still hasn't arrived. "
    "The tracking says 'in transit' since day one. I contacted support twice and got no reply. "
    "I need this for work urgently. Very frustrated."
)

extracted = extract_review_data(test_review)
print(json.dumps(extracted, indent=2))

---

## Part 6 — Simple Review Q&A Assistant

Build a chatbot that can answer questions about a set of reviews (a minimal RAG-style approach).

In [ ]:
review_corpus = """
REVIEW 001 | POSITIVE: "Fast delivery, arrived in 2 days. Product is exactly as described."
REVIEW 002 | NEGATIVE: "Delivery took 3 weeks and arrived damaged. Support didn't help."
REVIEW 003 | POSITIVE: "Amazing customer service. Solved my issue within hours."
REVIEW 004 | NEGATIVE: "Product broke after one week. Very cheap materials."
REVIEW 005 | POSITIVE: "Great value for money. Would definitely buy again."
REVIEW 006 | NEGATIVE: "Incorrect item delivered. Still waiting for a refund after 2 weeks."
REVIEW 007 | POSITIVE: "Excellent quality and packaging. Very impressed."
REVIEW 008 | NEUTRAL: "Product is okay, nothing special. Delivery was on time."
"""

def review_qa(question: str) -> str:
    prompt = f"""
You are a data analyst assistant. Use ONLY the customer reviews provided below to answer the question.
If the answer cannot be found in the reviews, say so clearly.

Customer Reviews:
{review_corpus}

Question: {question}
"""
    return ask(prompt, temperature=0.1)


questions = [
    "What are the most common complaints in these reviews?",
    "How many reviews mention delivery problems?",
    "What do customers say about customer support?",
]

for q in questions:
    print(f'Q: {q}')
    print(f'A: {review_qa(q)}')
    print()

---

## Part 7 — Classic NLP vs Gen AI: Side-by-Side

| | Classic NLP | Gen AI (LLM) |
|---|---|---|
| **Training data needed** | Yes (labelled) | No (zero-shot) |
| **Nuance / context** | Limited | Excellent |
| **Speed** | Very fast (ms) | Slower (API call) |
| **Cost** | Near-zero (local) | API cost per token |
| **Explainability** | Easy (coefficients) | Harder |
| **Multilingual** | Needs separate models | Often built-in |
| **Best for** | High-volume simple tasks | Complex, flexible tasks |

**Rule of thumb:** Start with classic NLP if you have labelled data and need scale. Use LLMs when you need flexibility, zero-shot capability, or human-like understanding.

---

## Your Turn! 🧪

**Exercise A — Prompting:**
Modify the sentiment classification prompt to also detect the **emotion** in the review (e.g. "anger", "joy", "frustration", "satisfaction"). Add this as a new key in the JSON output.

**Exercise B — Chain-of-Thought:**
Add the instruction `"Think step by step before answering"` to the sentiment prompt. Does the output change? Is it more accurate for ambiguous reviews?

**Exercise C — Extraction pipeline:**
Apply `extract_review_data()` to all 8 reviews in the corpus above. Build a DataFrame from the results. Which reviews have `urgency = "high"`?

**Exercise D (Bonus) — Multi-turn conversation:**
Extend the `ask()` function to support a list of messages (conversation history) instead of a single prompt. Build a small chat loop that lets you ask multiple follow-up questions about the reviews.

In [ ]:
# Your code here


---

## Workshop Wrap-Up

Over the 4 sessions we built a complete ML & AI toolkit using a single business story:

| Session | Technique | Business Problem |
|---|---|---|
| 01 | Regression + Classification | Predict spend & churn |
| 02 | Clustering + Anomaly Detection | Segment customers, detect fraud |
| 03 | Classic NLP | Classify review sentiment |
| 04 | Gen AI / LLMs | Extract insights, summarise, Q&A |

### Key Takeaways

1. **Choose the right tool for the job** — not every problem needs an LLM.
2. **Data quality matters more than model choice** — garbage in, garbage out.
3. **Evaluation is non-negotiable** — always measure before deploying.
4. **Business context drives decisions** — a metric like precision vs recall depends on the cost of each type of error.
5. **LLMs are powerful but have tradeoffs** — speed, cost, and explainability matter in production.

### Resources

- [scikit-learn documentation](https://scikit-learn.org/stable/)
- [Anthropic API documentation](https://docs.anthropic.com/)
- [Prompt engineering guide — Anthropic](https://docs.anthropic.com/en/docs/build-with-claude/prompt-engineering/overview)